In [6]:

!pip install transformers datasets scikit-learn -q
!pip install git+https://github.com/csebuetnlp/normalizer -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [7]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

In [8]:
BASE_MODEL  = "csebuetnlp/banglabert"
DATA_CSV    = "ai_human_dataset.csv"
OUTPUT_DIR  = "bangla_ai_detector"

MAX_LENGTH  = 512
BATCH_SIZE  = 8
EPOCHS      = 5
LR          = 2e-5
RANDOM_SEED = 42

In [9]:
df = pd.read_csv(DATA_CSV)

train_df = df[df["Split"] == "train"][["Text", "Label"]].reset_index(drop=True)
val_df   = df[df["Split"] == "val"][["Text", "Label"]].reset_index(drop=True)
test_df  = df[df["Split"] == "test"][["Text", "Label"]].reset_index(drop=True)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")

train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)
test_ds  = Dataset.from_pandas(test_df)

Train: 70  |  Val: 15  |  Test: 15


In [10]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(
        batch["Text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

tokenized_train = train_ds.map(tokenize, batched=True, remove_columns=["Text"])
tokenized_val   = val_ds.map(tokenize,   batched=True, remove_columns=["Text"])
tokenized_test  = test_ds.map(tokenize,  batched=True, remove_columns=["Text"])

tokenized_train = tokenized_train.rename_column("Label", "labels")
tokenized_val   = tokenized_val.rename_column("Label",   "labels")
tokenized_test  = tokenized_test.rename_column("Label",  "labels")

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/70 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "Human", 1: "AI"},
    label2id={"Human": 0, "AI": 1},
)
print(f"Model parameters: {model.num_parameters():,}")

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] ElectraForSequenceClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 110,618,882


In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [13]:
args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    fp16                        = True,    # BERT is stable with fp16
    logging_steps               = 10,
    save_total_limit            = 2,
    report_to                   = "none",
    seed                        = RANDOM_SEED,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
trainer = Trainer(
    model            = model,
    args             = args,
    train_dataset    = tokenized_train,
    eval_dataset     = tokenized_val,
    processing_class = tokenizer,
    compute_metrics  = compute_metrics,
    callbacks        = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Starting BanglaBERT fine-tuning...")
trainer.train()

Starting BanglaBERT fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.615462,1.000000,1.000000,1.000000,1.000000
2,0.663898,0.433805,1.000000,1.000000,1.000000,1.000000
3,0.513461,0.305029,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=27, training_loss=0.5313735802968343, metrics={'train_runtime': 79.3353, 'train_samples_per_second': 4.412, 'train_steps_per_second': 0.567, 'total_flos': 33363715152720.0, 'train_loss': 0.5313735802968343, 'epoch': 3.0})

In [15]:
import os
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\nModel saved to '{OUTPUT_DIR}/'")
print(f"Files: {os.listdir(OUTPUT_DIR)}")

# ── Cell 11: Evaluate on test set ─────────────────────────────────────────────
print("\n--- Test Set Evaluation ---")
results = trainer.predict(tokenized_test)
preds   = np.argmax(results.predictions, axis=-1)
labels  = results.label_ids

print(classification_report(labels, preds, target_names=["Human", "AI"]))
print("Confusion Matrix:")
print(confusion_matrix(labels, preds))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to 'bangla_ai_detector/'
Files: ['checkpoint-27', 'config.json', 'training_args.bin', 'tokenizer_config.json', 'tokenizer.json', 'checkpoint-9', 'model.safetensors']

--- Test Set Evaluation ---


              precision    recall  f1-score   support

       Human       1.00      1.00      1.00         8
          AI       1.00      1.00      1.00         7

    accuracy                           1.00        15
   macro avg       1.00      1.00      1.00        15
weighted avg       1.00      1.00      1.00        15

Confusion Matrix:
[[8 0]
 [0 7]]


In [16]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_SAVE_PATH = "/content/drive/MyDrive/bangla_ai_detector"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

for item in os.listdir(OUTPUT_DIR):
    if item.startswith("checkpoint-"):
        print(f"  Skipped: {item}")
        continue
    src  = os.path.join(OUTPUT_DIR, item)
    dest = os.path.join(DRIVE_SAVE_PATH, item)
    shutil.copy2(src, dest) if os.path.isfile(src) else shutil.copytree(src, dest, dirs_exist_ok=True)
    print(f"  Copied: {item}")

print(f"\nSaved to Google Drive: '{DRIVE_SAVE_PATH}'")

Mounted at /content/drive
  Skipped: checkpoint-27
  Copied: config.json
  Copied: training_args.bin
  Copied: tokenizer_config.json
  Copied: tokenizer.json
  Skipped: checkpoint-9
  Copied: model.safetensors

Saved to Google Drive: '/content/drive/MyDrive/bangla_ai_detector'
